# Dynamical and hybrid conditioning for compositional image retrieval

1. Download the necessary libraries
2. Setup the CelebA dataset
3. Download the CLIP model off of HuggingFace
4. Compute and save to disk the dataset embeddings
5. Define the baseline retrieval function
6. Define the retrieval metrics
7. Evaluate performance


## Download the necessary libraries

In [ ]:
%pip install -q torch torchvision tensorboard ftfy regex tqdm scikit-learn scikit-image

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.7 MB/s eta 0:00:00


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from PIL import Image
import skimage
import torch
import torch.nn as nn
import torchvision
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

## Setup the CelebA dataset

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!mkdir /content/datasets

In [ ]:
# This should take 1-2 minutes
# It unzips the dataset in the runtime's local SSD, so when
# you disconnect, it gets deleted
!unzip -q /content/drive/MyDrive/datasets/celeba.zip -d /content/datasets/

In [ ]:
import torch
from pathlib import Path

from torchvision.datasets import CelebA

In [ ]:
# Do *not* put `celeba` in the path.
# The dataset class will do that automatically!
data_root = Path("/content/datasets")

In [ ]:
celeba = CelebA(root=data_root, split="test", download=False)

In [ ]:
# This should be 19,962
len(celeba)

19962

## Download the CLIP model off of HuggingFace

In [ ]:
from transformers import AutoProcessor, AutoModel

# Download the CLIP ViT-B/32 model
model_name = "openai/clip-vit-base-patch32"

# Load the processor (tokenizer and image preprocessor)
processor = AutoProcessor.from_pretrained(model_name)

# Load the model
model = AutoModel.from_pretrained(model_name)

# Set the model to eval mode
model = model.cuda().eval()

print(f"Successfully downloaded and loaded {model_name} model and processor.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Successfully downloaded and loaded openai/clip-vit-base-patch32 model and processor.


## Compute and save to disk the dataset embeddings

In [ ]:
import torch
import os
from tqdm import tqdm

# Due to the constraint of modifying only this cell, the 'encode_celeba_images' function
# is redefined here with the fix, even though its original definition is in another cell.
def encode_celeba_images(celeba_dataset, processor, model, device="cuda"):

    all_image_embeddings = []
    model.to(device) # Ensure model is on the correct device
    model.eval()     # Set model to evaluation mode

    # Use tqdm for a progress bar to show progress during encoding
    for i in tqdm(range(len(celeba_dataset)), desc="Encoding CelebA images"):
        image, _ = celeba_dataset[i] # CelebA returns (image, target)

        # Preprocess the image using the processor from transformers
        # The processor handles transformations like resizing, normalization, etc.
        inputs = processor(images=image, return_tensors="pt").to(device)

        # Encode the image using the model from transformers
        with torch.no_grad():
            image_features = model.get_image_features(pixel_values=inputs.pixel_values)

        # FIX: Access .pooler_output as image_features is BaseModelOutputWithPooling
        all_image_embeddings.append(image_features.pooler_output.cpu()) # Move to CPU to prevent GPU memory issues during accumulation

    return torch.cat(all_image_embeddings, dim=0)

# Call the function to encode CelebA images
celeba_image_embeddings = encode_celeba_images(celeba_dataset=celeba, processor=processor, model=model, device="cuda")

# Define paths for saving
local_save_path = "/content/celeba_image_embeddings.pt"
drive_save_path = "/content/drive/MyDrive/datasets/celeba_image_embeddings.pt"

# Save locally
torch.save(celeba_image_embeddings, local_save_path)
print(f"Image embeddings saved locally to: {local_save_path}")

# Ensure the directory exists in Google Drive before saving
drive_dir = os.path.dirname(drive_save_path)
if not os.path.exists(drive_dir):
    os.makedirs(drive_dir)
    print(f"Created directory on Google Drive: {drive_dir}")

# Save to Google Drive
torch.save(celeba_image_embeddings, drive_save_path)
print(f"Image embeddings saved to Google Drive: {drive_save_path}")

Encoding CelebA images:   2%|▏         | 486/19962 [00:08<05:57, 54.43it/s]


KeyboardInterrupt: 

In [ ]:
import torch

def get_text_embedding(text, processor, model, device="cuda"):
    """
    Generates CLIP text embedding for the given text.
    (Fixed for BaseModelOutputWithPooling output and now returns [512] shape)
    """
    model.to(device)
    model.eval()
    inputs = processor(text=text, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        text_features = model.get_text_features(**inputs)
    # FIX: Access .pooler_output and then squeeze to remove the batch dimension
    return text_features.pooler_output.squeeze(0).cpu()

## Define the baseline retrieval function

In [ ]:
import json

annotations_path = Path("/content/drive/MyDrive/datasets/celeba_evaluation.json")

with open(annotations_path, "r") as f:
    annotations = json.load(f)

len(annotations)

14

In [ ]:
import torch
import torch.nn.functional as F

def simple_retrieval(query_index, query_source_image_index, celeba_image_embeddings, K):

  if type(query_source_image_index) is int:
    query_source_image_index = str(query_source_image_index)

  # Check that the source image index belongs to the example in that query
  # so that we will then be able to check its corresponding ground truth
  try:
    value = annotations[query_index]['ground_truth'][query_source_image_index]
  except (IndexError, KeyError, TypeError) as e:
    return {"error": str(e)}

  # get the original image embedding
  # This will have shape [512]
  query_source_image_embedding = celeba_image_embeddings[int(query_source_image_index)]

  # normalize source image embedding for stability in CLIP space
  query_source_image_embedding = F.normalize(query_source_image_embedding, dim=0)

  computed_embedding = query_source_image_embedding.clone()

  for attribute in annotations[query_index]['query'].split(','):

    # extract the correct attribute
    attribute = attribute.strip()
    is_it_positive = attribute[0] =='+'
    attribute = attribute[1:]

    # attribute_embedding will now have shape [512] due to .squeeze(0) in get_text_embedding
    attribute_embedding = get_text_embedding(attribute, processor, model)

    # normalize each attribute embedding for consistent CLIP space arithmetic
    attribute_embedding = F.normalize(attribute_embedding, dim=0)

    if is_it_positive:
        computed_embedding += attribute_embedding
    else:
        computed_embedding -= attribute_embedding

  # Normalize the computed embedding
  normalized_computed_embedding = F.normalize(computed_embedding.unsqueeze(0), p=2, dim=1)

  # Normalize all celeba image embeddings
  normalized_celeba_embeddings = F.normalize(celeba_image_embeddings, p=2, dim=1)

  # Compute cosine similarity between the computed_embedding and all celeba_image_embeddings
  # The result will be a tensor of shape [1, num_images]
  similarities = torch.mm(normalized_computed_embedding, normalized_celeba_embeddings.T)

  # Get the top K most similar images (indices)
  # .squeeze() converts [1, num_images] to [num_images]
  # .topk(K) returns a tuple (values, indices)
  top_k_similarities, top_k_indices = torch.topk(similarities.squeeze(), K)

  # Return the indices of the K most similar images
  return top_k_indices.tolist()

In [ ]:
# Original call
simple_retrieval(12, 37, celeba_image_embeddings, 10)

[37, 16325, 13, 8583, 19354, 19194, 10317, 15905, 13042, 6993]

In [ ]:
simple_retrieval(0, "13", celeba_image_embeddings, 10)

[13, 19005, 9619, 4650, 3874, 13877, 16927, 7357, 8348, 17732]

## Define the retrieval metrics

In [ ]:
def evaluate_retrieval(
    retrieved_indices: list[int],
    ground_truth_indices: list[int],
    k: int
):
    """
    Evaluate the retrieval performance for a single source image.

    Args:
    ----
        retrieved_indices: list of image IDs predicted by the model,
            ordered by similarity (descending).
        ground_truth_indices: list of valid target IDs from the benchmark JSON.
        k: the cutoff for top-K evaluation (e.g., 1, 5, 10).

    Return:
    ------
        A dictionary containing Recall@K and Precision@K.

    """
    # Isolate the top K predictions
    top_k_retrieved = retrieved_indices[:k]

    # Calculate the intersection between predictions and ground truth
    hits = set(top_k_retrieved).intersection(set(ground_truth_indices))
    num_hits = len(hits)

    # Metrics calculations
    # Recall@K (Hit Rate): 1 if at least one match is found, 0 otherwise
    recall_at_k = 1 if num_hits > 0 else 0

    # Precision@K: Fraction of top K predictions that are correct
    precision_at_k = num_hits / k

    return {
        f"Recall@{k}": recall_at_k,
        f"Precision@{k}": precision_at_k
    }

# --- Example Usage ---
# Suppose the model returns these indices from most to least similar:
# predictions = [1, 2, 3, 4, 5]
# And we load this from our JSON for this specific source:
# ground_truth = [3, 2, 1]

# Evaluate at K=1 and K=5
# print("Results @ 1:", evaluate_retrieval(predictions, ground_truth, k=1))
# print("Results @ 5:", evaluate_retrieval(predictions, ground_truth, k=5))

## Evaluate performance

In [ ]:
def metrics_per_query(query_index, celeba_image_embeddings, K):
  total_recall = 0.0
  total_precision = 0.0
  evaluation_count = 0

  for test_image_index, _ in annotations[query_index]['ground_truth'].items():
      predictions = simple_retrieval(query_index, test_image_index, celeba_image_embeddings, K)

      # Handle potential errors from simple_retrieval
      if isinstance(predictions, dict) and "error" in predictions:
            print(f"Warning: simple_retrieval returned an error for query_index {query_index}, source image '{test_image_index}': {predictions['error']}")
            continue # Skip this evaluation if an error occurred

      metrics = evaluate_retrieval(predictions, annotations[query_index]['ground_truth'][str(test_image_index)], K)

      total_recall += metrics[f"Recall@{K}"]
      total_precision += metrics[f"Precision@{K}"]
      evaluation_count += 1

  if evaluation_count > 0:
      average_recall = total_recall / evaluation_count
      average_precision = total_precision / evaluation_count
      print(f"Query Index {query_index} (K={K}):")
      print(f"  Average Recall@{K}: {average_recall:.4f}")
      print(f"  Average Precision@{K}: {average_precision:.4f}")
  else:
      print(f"Query Index {query_index} (K={K}): No evaluations performed.")

In [ ]:
metrics_per_query(0, celeba_image_embeddings, 10)

Query Index 0 (K=10):
  Average Recall@10: 0.0000
  Average Precision@10: 0.0000


In [ ]:
for query_index in range(1, len(annotations)):
  metrics_per_query(query_index, celeba_image_embeddings, 10)

Query Index 1 (K=10):
  Average Recall@10: 0.0036
  Average Precision@10: 0.0005
Query Index 2 (K=10):
  Average Recall@10: 0.0039
  Average Precision@10: 0.0007
Query Index 3 (K=10):
  Average Recall@10: 0.0025
  Average Precision@10: 0.0003
Query Index 4 (K=10):
  Average Recall@10: 0.0063
  Average Precision@10: 0.0009
Query Index 5 (K=10):
  Average Recall@10: 0.0057
  Average Precision@10: 0.0006
Query Index 6 (K=10):
  Average Recall@10: 0.0299
  Average Precision@10: 0.0030
Query Index 7 (K=10):
  Average Recall@10: 0.0000
  Average Precision@10: 0.0000
Query Index 8 (K=10):
  Average Recall@10: 0.0000
  Average Precision@10: 0.0000
Query Index 9 (K=10):
  Average Recall@10: 0.0432
  Average Precision@10: 0.0049
Query Index 10 (K=10):
  Average Recall@10: 0.0000
  Average Precision@10: 0.0000
Query Index 11 (K=10):
  Average Recall@10: 0.0034
  Average Precision@10: 0.0003
Query Index 12 (K=10):
  Average Recall@10: 0.0000
  Average Precision@10: 0.0000
Query Index 13 (K=10):
  